# Task 4 — Biological Interpretation

Analyze network properties, interpret biological significance of hub and bottleneck nodes, and propose drug repurposing candidates with strong biological justification based on network metrics and pathway analysis.

## 1. Initialize Project Environment

In [10]:
"""Setup and imports for Lab 9 Task 4 - Biological Interpretation."""
import logging
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List

import networkx as nx
import pandas as pd

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

print(f"Python {sys.version}")
print(f"networkx {nx.__version__}")

Python 3.12.3 (main, Jan  8 2026, 11:30:50) [GCC 13.3.0]
networkx 3.3


## 2. Define Configuration Parameters

In [11]:
@dataclass
class Task4Config:
    """Configuration for biological interpretation."""

    handle: str = "AndreiCod"
    export_dir: Path = Path("./artifacts")
    network_file: str = "task2_network.gml"
    metrics_file: str = "task3_centrality_metrics.csv"
    paths_file: str = "task3_drug_disease_paths.csv"
    hub_targets_file: str = "task3_hub_targets.csv"

    def __post_init__(self):
        self.export_dir.mkdir(parents=True, exist_ok=True)

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = Task4Config()
CONFIG.describe()

{'handle': 'AndreiCod',
 'export_dir': 'artifacts',
 'network_file': 'task2_network.gml',
 'metrics_file': 'task3_centrality_metrics.csv',
 'paths_file': 'task3_drug_disease_paths.csv',
 'hub_targets_file': 'task3_hub_targets.csv'}

## 3. Implement Core Functionality

In [12]:
# Load network and metrics from previous tasks
G = nx.read_gml(CONFIG.export_dir / CONFIG.network_file)
metrics_df = pd.read_csv(CONFIG.export_dir / CONFIG.metrics_file)
paths_df = pd.read_csv(CONFIG.export_dir / CONFIG.paths_file)
hub_targets_df = pd.read_csv(CONFIG.export_dir / CONFIG.hub_targets_file)

logging.info(
    "Loaded network: %d nodes, %d edges", G.number_of_nodes(), G.number_of_edges()
)
logging.info("Loaded metrics for %d nodes", len(metrics_df))
logging.info("Loaded %d drug-disease paths", len(paths_df))
print(f"[OK] All data loaded successfully")

23:21:16 | INFO | Loaded network: 26616 nodes, 38150 edges
23:21:16 | INFO | Loaded metrics for 26616 nodes
23:21:16 | INFO | Loaded 5000 drug-disease paths


[OK] All data loaded successfully


In [13]:
def analyze_network_properties(G: nx.Graph) -> Dict:
    """Compute global network properties with biological interpretation."""
    properties = {
        "num_nodes": G.number_of_nodes(),
        "num_edges": G.number_of_edges(),
        "density": nx.density(G),
        "num_connected_components": nx.number_connected_components(G),
        "avg_clustering": nx.average_clustering(G),
        "avg_degree": sum(dict(G.degree()).values()) / G.number_of_nodes(),
    }

    # Get largest connected component size
    largest_cc = max(nx.connected_components(G), key=len)
    properties["largest_component_size"] = len(largest_cc)
    properties["largest_component_pct"] = len(largest_cc) / G.number_of_nodes() * 100

    # Node type counts (now 4 types)
    for node_type in ["Drug", "Target", "Gene", "Disease"]:
        count = len(
            [n for n, d in G.nodes(data=True) if d.get("node_type") == node_type]
        )
        properties[f"{node_type.lower()}_nodes"] = count

    # Edge type counts
    edge_counts = {}
    for u, v, d in G.edges(data=True):
        edge_type = d.get("edge_type", "unknown")
        edge_counts[edge_type] = edge_counts.get(edge_type, 0) + 1
    for etype, count in edge_counts.items():
        properties[f"{etype.replace('-', '_')}_edges"] = count

    logging.info("Network properties computed")
    return properties


# Analyze network
network_props = analyze_network_properties(G)
print("\n=== Network Properties ===")
for key, value in network_props.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

23:21:16 | INFO | Network properties computed



=== Network Properties ===
  num_nodes: 26616
  num_edges: 38150
  density: 0.0001
  num_connected_components: 10366
  avg_clustering: 0.0000
  avg_degree: 2.8667
  largest_component_size: 14564
  largest_component_pct: 54.7190
  drug_nodes: 16575
  target_nodes: 2887
  gene_nodes: 2794
  disease_nodes: 4360
  drug_target_edges: 14802
  drug_disease_edges: 5983
  protein_gene_edges: 2794
  gene_disease_edges: 14571


In [14]:
def interpret_centrality(metrics_df: pd.DataFrame) -> Dict:
    """Interpret centrality metrics for biological insights."""
    interpretation = {}

    # Get top hubs by node type
    for node_type in ["Drug", "Target", "Disease"]:
        type_df = metrics_df[metrics_df["node_type"] == node_type]
        top_degree = type_df.nlargest(5, "degree_centrality")
        top_betweenness = type_df.nlargest(5, "betweenness_centrality")

        interpretation[f"top_{node_type.lower()}_hubs"] = top_degree["node"].tolist()
        interpretation[f"top_{node_type.lower()}_bottlenecks"] = top_betweenness[
            "node"
        ].tolist()

    # Overall statistics
    interpretation["avg_clustering"] = metrics_df["clustering_coefficient"].mean()
    interpretation["max_degree"] = metrics_df["degree"].max()
    interpretation["nodes_with_high_betweenness"] = len(
        metrics_df[metrics_df["betweenness_centrality"] > 0.01]
    )

    return interpretation


# Interpret centrality
centrality_interp = interpret_centrality(metrics_df)

print("\n=== Centrality Interpretation ===")
print(f"\nTop Drug Hubs (most targets/diseases):")
for drug in centrality_interp["top_drug_hubs"][:5]:
    print(f"  - {drug}")

print(f"\nTop Target Hubs (most drug interactions):")
for target in centrality_interp["top_target_hubs"][:5]:
    print(f"  - {target}")

print(f"\nTop Disease Hubs (most drug treatments):")
for disease in centrality_interp["top_disease_hubs"][:5]:
    print(f"  - {disease}")


=== Centrality Interpretation ===

Top Drug Hubs (most targets/diseases):
  - Fostamatinib
  - NADH
  - Copper
  - Zinc
  - Zinc acetate

Top Target Hubs (most drug interactions):
  - Cyclin-dependent kinase 2
  - Estrogen receptor alpha
  - Histamine H1 receptor
  - Dopamine D2 receptor
  - 5-hydroxytryptamine receptor 2A

Top Disease Hubs (most drug treatments):
  - Syndrome, Alzheimer'S Disease
  - Parkinson'S Disease, Chronic
  - Solid Tumors
  - Zinc Deficiency/Its Consequences
  - Ear Infections, As Well


## 4. Validate with Unit Tests

In [15]:
# Validate interpretation
assert len(centrality_interp["top_drug_hubs"]) > 0, "No drug hubs found"
assert len(centrality_interp["top_target_hubs"]) > 0, "No target hubs found"
assert network_props["num_nodes"] > 0, "Empty network"
assert network_props["density"] > 0, "Zero density"

print("[OK] Network interpretation validation passed")

[OK] Network interpretation validation passed


In [ ]:
def propose_repurposing_candidates_shared_target(
    G: nx.Graph,
    metrics_df: pd.DataFrame,
    top_n: int = 10,
) -> pd.DataFrame:
    """
    Propose drug repurposing candidates based on SHARED TARGET mechanism.

    VALID approach (no circular reasoning):
    - If Drug A and Drug B both bind Target T
    - And Drug A is approved for Disease D
    - Then Drug B may also work for Disease D via the same mechanism

    This uses ONLY native DrugBank relationships:
    - Drug → Target (direct binding)
    - Drug → Disease (direct indication)

    NO inferred gene-disease edges are used.
    """
    from collections import defaultdict

    # Build drug -> targets mapping (from network edges)
    drug_targets = defaultdict(set)
    target_drugs = defaultdict(set)

    for drug in G.nodes():
        if G.nodes[drug].get("node_type") != "Drug":
            continue
        for neighbor in G.neighbors(drug):
            if G.nodes[neighbor].get("node_type") == "Target":
                edge_data = G.get_edge_data(drug, neighbor)
                if edge_data and edge_data.get("edge_type") == "drug-target":
                    drug_targets[drug].add(neighbor)
                    target_drugs[neighbor].add(drug)

    # Build drug -> diseases mapping (direct indications only)
    drug_diseases = defaultdict(set)
    for drug in G.nodes():
        if G.nodes[drug].get("node_type") != "Drug":
            continue
        for neighbor in G.neighbors(drug):
            if G.nodes[neighbor].get("node_type") == "Disease":
                edge_data = G.get_edge_data(drug, neighbor)
                if edge_data and edge_data.get("edge_type") == "drug-disease":
                    drug_diseases[drug].add(neighbor)

    logging.info(f"Found {len(drug_targets)} drugs with targets, {len(target_drugs)} targets with drugs")
    logging.info(f"Found {len([d for d in drug_diseases if drug_diseases[d]])} drugs with indications")

    # Find repurposing candidates via shared targets
    candidates = []
    seen_pairs = set()  # Avoid duplicates

    for target, drugs in target_drugs.items():
        if len(drugs) < 2:
            continue
        drugs = list(drugs)

        for i, drug_a in enumerate(drugs):
            for drug_b in drugs[i+1:]:
                # Skip if neither has indications
                if not drug_diseases[drug_a] and not drug_diseases[drug_b]:
                    continue

                # Get metrics for scoring
                drug_b_metrics = metrics_df[metrics_df["node"] == drug_b]
                drug_a_metrics = metrics_df[metrics_df["node"] == drug_a]

                # Diseases drug_a treats but drug_b doesn't
                diseases_a_only = drug_diseases[drug_a] - drug_diseases[drug_b]
                for disease in diseases_a_only:
                    pair_key = (drug_b, disease)
                    if pair_key in seen_pairs:
                        continue
                    seen_pairs.add(pair_key)

                    # Calculate score based on target connectivity
                    betweenness = drug_b_metrics["betweenness_centrality"].values[0] if len(drug_b_metrics) > 0 else 0
                    degree = drug_b_metrics["degree"].values[0] if len(drug_b_metrics) > 0 else 0

                    # Score: path length is 3 (drug_b -> target -> drug_a -> disease)
                    score = 1.0/3 * (1 + betweenness * 10) + 0.3  # mechanistic bonus

                    candidates.append({
                        "drug": drug_b,
                        "disease": disease,
                        "path_length": 3,
                        "path": f"{drug_b} → {target} ← {drug_a} → {disease}",
                        "shared_target": target,
                        "approved_drug": drug_a,
                        "drug_degree": degree,
                        "drug_betweenness": betweenness,
                        "has_mechanistic_path": True,
                        "repurposing_score": score,
                    })

                # Diseases drug_b treats but drug_a doesn't
                diseases_b_only = drug_diseases[drug_b] - drug_diseases[drug_a]
                for disease in diseases_b_only:
                    pair_key = (drug_a, disease)
                    if pair_key in seen_pairs:
                        continue
                    seen_pairs.add(pair_key)

                    betweenness = drug_a_metrics["betweenness_centrality"].values[0] if len(drug_a_metrics) > 0 else 0
                    degree = drug_a_metrics["degree"].values[0] if len(drug_a_metrics) > 0 else 0

                    score = 1.0/3 * (1 + betweenness * 10) + 0.3

                    candidates.append({
                        "drug": drug_a,
                        "disease": disease,
                        "path_length": 3,
                        "path": f"{drug_a} → {target} ← {drug_b} → {disease}",
                        "shared_target": target,
                        "approved_drug": drug_b,
                        "drug_degree": degree,
                        "drug_betweenness": betweenness,
                        "has_mechanistic_path": True,
                        "repurposing_score": score,
                    })

    logging.info(f"Found {len(candidates)} total shared-target repurposing candidates")

    if not candidates:
        logging.warning("No repurposing candidates found")
        return pd.DataFrame()

    candidates_df = pd.DataFrame(candidates)
    candidates_df = candidates_df.sort_values("repurposing_score", ascending=False)

    # Select diverse candidates (different drugs and diseases)
    diverse_candidates = []
    drugs_selected = {}
    diseases_selected = {}
    targets_selected = {}

    for _, row in candidates_df.iterrows():
        drug = row["drug"]
        disease = row["disease"]
        target = row["shared_target"]

        drug_count = drugs_selected.get(drug, 0)
        disease_count = diseases_selected.get(disease, 0)
        target_count = targets_selected.get(target, 0)

        # Max 1 per drug, 1 per disease, 2 per target for diversity
        if drug_count < 1 and disease_count < 1 and target_count < 2:
            diverse_candidates.append(row.to_dict())
            drugs_selected[drug] = drug_count + 1
            diseases_selected[disease] = disease_count + 1
            targets_selected[target] = target_count + 1

        if len(diverse_candidates) >= top_n:
            break

    result_df = pd.DataFrame(diverse_candidates)
    logging.info(f"Selected {len(result_df)} diverse repurposing candidates")
    return result_df


# Propose repurposing candidates using SHARED TARGET approach (no circular reasoning)
repurposing_df = propose_repurposing_candidates_shared_target(G, metrics_df, top_n=10)
print(f"\n=== Top Drug Repurposing Candidates (Shared Target Mechanism) ===")
print(f"Found {len(repurposing_df)} diverse candidates\n")

if len(repurposing_df) > 0:
    # Show top candidates
    display(
        repurposing_df.head(10)[
            [
                "drug",
                "disease",
                "shared_target",
                "approved_drug",
                "repurposing_score",
            ]
        ]
    )

23:21:16 | INFO | Found 5871 drugs with targets, 2887 targets with drugs
23:21:16 | INFO | Found 2961 drugs with indications


## 5. Export Results

In [ ]:
# Export network summary with biological interpretation
EXPORT_DIR = CONFIG.export_dir
summary_file = EXPORT_DIR / "task4_network_summary.txt"

with open(summary_file, "w") as f:
    f.write("=" * 70 + "\n")
    f.write("DRUG-DISEASE NETWORK ANALYSIS SUMMARY\n")
    f.write("Assignment 8: Drug-Disease Network Construction and Analysis\n")
    f.write("=" * 70 + "\n\n")

    f.write("## Data Source\n")
    f.write("DrugBank XML v5.1.11 (full dataset)\n\n")

    f.write("## Network Properties\n")
    for key, value in network_props.items():
        if isinstance(value, float):
            f.write(f"  {key}: {value:.4f}\n")
        else:
            f.write(f"  {key}: {value}\n")

    f.write("\n## Network Interpretation\n")

    # Density interpretation
    density = network_props["density"]
    if density < 0.001:
        f.write(
            "- VERY SPARSE network (density < 0.001): Highly selective drug-target binding.\n"
        )
        f.write(
            "  This is typical of drug-protein interaction networks where drugs bind specific targets.\n"
        )
    elif density < 0.01:
        f.write(
            "- Sparse network: Selective drug-target interactions typical of biological networks.\n"
        )
    else:
        f.write(
            "- Moderately dense network: Some degree of polypharmacology (multi-target drugs).\n"
        )

    # Connectivity interpretation
    cc_pct = network_props["largest_component_pct"]
    if cc_pct > 90:
        f.write(f"- WELL-CONNECTED: {cc_pct:.1f}% of nodes in largest component.\n")
        f.write(
            "  Most drugs, targets, and diseases are interconnected, enabling path-based analysis.\n"
        )
    elif cc_pct > 50:
        f.write(f"- Moderately connected: {cc_pct:.1f}% in largest component.\n")
    else:
        f.write(f"- FRAGMENTED network: Only {cc_pct:.1f}% in largest component.\n")
        f.write("  Many disconnected drug-disease clusters.\n")

    # Clustering interpretation
    avg_clust = network_props["avg_clustering"]
    if avg_clust < 0.01:
        f.write(
            f"- Very low clustering ({avg_clust:.4f}): Expected for bipartite-like structure.\n"
        )
        f.write(
            "  Drugs connect to targets, targets to diseases, but limited drug-drug or disease-disease connections.\n"
        )
    else:
        f.write(f"- Clustering coefficient: {avg_clust:.4f}\n")

    f.write("\n## Top Hub Drugs (Most Targets) - Key Polypharmacological Compounds\n")
    for drug in centrality_interp.get("top_drug_hubs", [])[:5]:
        drug_metrics = metrics_df[metrics_df["node"] == drug]
        if len(drug_metrics) > 0:
            deg = drug_metrics["degree"].values[0]
            btw = drug_metrics["betweenness_centrality"].values[0]
            f.write(f"  - {drug}: {deg} connections, betweenness={btw:.4f}\n")

    f.write("\n## Top Hub Targets (Most Drug Interactions) - Druggable Proteins\n")
    for target in centrality_interp.get("top_target_hubs", [])[:5]:
        target_metrics = metrics_df[metrics_df["node"] == target]
        if len(target_metrics) > 0:
            deg = target_metrics["degree"].values[0]
            btw = target_metrics["betweenness_centrality"].values[0]
            f.write(f"  - {target}: {deg} drugs, betweenness={btw:.4f}\n")

    f.write("\n## Top Bottleneck Nodes (High Betweenness) - Critical Regulators\n")
    for node in centrality_interp.get("top_target_bottlenecks", [])[:5]:
        node_metrics = metrics_df[metrics_df["node"] == node]
        if len(node_metrics) > 0:
            btw = node_metrics["betweenness_centrality"].values[0]
            ntype = node_metrics["node_type"].values[0]
            f.write(f"  - {node} ({ntype}): betweenness={btw:.4f}\n")

    f.write("\n## Drug Repurposing Analysis\n")
    f.write(
        "Strategy: SHARED TARGET MECHANISM (no circular reasoning)\n"
    )
    f.write(
        "If Drug A and Drug B both bind Target T, and Drug A treats Disease D,\n"
    )
    f.write(
        "then Drug B may also treat Disease D via the same mechanistic pathway.\n\n"
    )

    if len(repurposing_df) > 0:
        f.write("### Top Repurposing Candidates:\n")
        for i, (_, row) in enumerate(repurposing_df.head(5).iterrows(), 1):
            f.write(f"\n{i}. {row['drug']} → {row['disease']}\n")
            f.write(f"   Shared Target: {row['shared_target']}\n")
            f.write(f"   Approved Drug: {row['approved_drug']} (proven for this disease)\n")
            f.write(f"   Path: {row['path']}\n")
            f.write(f"   Score: {row['repurposing_score']:.4f}\n")

print(f"[OK] Network summary saved to: {summary_file.resolve()}")

[OK] Network summary saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/09_repurposing/assignments/artifacts/task4_network_summary.txt


In [ ]:
# Export repurposing candidates with justification
candidates_file = EXPORT_DIR / "task4_repurposing_candidates.csv"

if len(repurposing_df) > 0:
    # Add biological justification column for SHARED TARGET approach
    def generate_justification(row):
        """Generate biological justification for shared-target repurposing candidate."""
        drug = row["drug"]
        disease = row["disease"]
        shared_target = row.get("shared_target", "unknown target")
        approved_drug = row.get("approved_drug", "another drug")

        return (
            f"{drug} and {approved_drug} both bind {shared_target}. "
            f"Since {approved_drug} is approved for {disease}, "
            f"{drug} may also be effective via the same mechanism."
        )

    repurposing_df["justification"] = repurposing_df.apply(
        generate_justification, axis=1
    )
    repurposing_df.to_csv(candidates_file, index=False)
else:
    pd.DataFrame().to_csv(candidates_file, index=False)

# Export network properties as CSV
props_file = EXPORT_DIR / "task4_network_properties.csv"
props_df = pd.DataFrame([network_props])
props_df.to_csv(props_file, index=False)

print(f"[OK] Repurposing candidates saved to: {candidates_file.resolve()}")
print(f"[OK] Network properties saved to: {props_file.resolve()}")

# Display final candidates with justification
if len(repurposing_df) > 0:
    print("\n=== Final Repurposing Candidates with Justification ===")
    for i, (_, row) in enumerate(repurposing_df.head(5).iterrows(), 1):
        print(f"\n{i}. {row['drug']} → {row['disease']}")
        print(f"   Shared Target: {row['shared_target']}")
        print(f"   Approved Drug: {row['approved_drug']}")
        print(f"   Justification: {row['justification']}")

print(f"\n[OK] Task 4 completed successfully!")

[OK] Repurposing candidates saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/09_repurposing/assignments/artifacts/task4_repurposing_candidates.csv
[OK] Network properties saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/09_repurposing/assignments/artifacts/task4_network_properties.csv

=== Final Repurposing Candidates with Justification ===

1. Cetuximab → Testicular Tumors
   Path: Cetuximab → High affinity immunoglobulin gamma Fc receptor I → Trastuzumab deruxtecan → Breast Cancer Who Have → Gene:TOP2A → Testicular Tumors
   Justification: Cetuximab binds High affinity immunoglobulin gamma Fc re, which is implicated in Testicular Tumors. Network analysis (path length=5) suggests shared mechanism.

2. Cetuximab → Newly-Diagnosed Therapy-Related Acute Myeloid Leukemia (T-Am
   Path: Cetuximab → High affinity immunoglobulin gamma Fc receptor I → Trastuzumab deruxtecan → Breast Cancer Who Have → Gene:TOP2A → Newly-Diagnosed Therapy-Related Acute Myeloid Leukemia (T-Am
   Justification: